# Stage 04: Data Acquisition and Ingestion

## Sources and parameters

- API: Alpha Vantage TIME_SERIES_DAILY, symbol IBM, compact output.
- API URL: https://www.alphavantage.co/query
- Scrape: first 10 rows of the S&P 500 component table.
- Scrape URL: https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
- Raw filenames use YYYYMMDD-HHMM timestamps.
- .env is excluded; only .env.example is committed.


In [1]:
from datetime import datetime
import os
from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    candidate = PROJECT_ROOT / "homework" / "homework04"
    if candidate.exists():
        PROJECT_ROOT = candidate
load_dotenv(PROJECT_ROOT / ".env")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RUN_TS = datetime.now().strftime("%Y%m%d-%H%M")
print("Raw directory:", RAW_DIR)
print("API key configured:", bool(os.getenv("ALPHAVANTAGE_API_KEY", "demo")))


Raw directory: homework/homework04/data/raw
API key configured: True


## 1. API ingestion


In [2]:
API_URL = "https://www.alphavantage.co/query"
params = {"function": "TIME_SERIES_DAILY", "symbol": "IBM",
          "outputsize": "compact",
          "apikey": os.getenv("ALPHAVANTAGE_API_KEY", "demo")}
try:
    response = requests.get(API_URL, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "Time Series (Daily)" not in payload:
        raise ValueError(payload.get("Note") or payload.get("Error Message") or "Unexpected API response")
    api_df = pd.DataFrame.from_dict(payload["Time Series (Daily)"], orient="index")
    api_df = api_df.rename(columns={"1. open":"open","2. high":"high","3. low":"low",
                                    "4. close":"close","5. volume":"volume"})
    api_df = api_df.rename_axis("date").reset_index()
    api_df["date"] = pd.to_datetime(api_df["date"])
    api_df[["open","high","low","close"]] = api_df[["open","high","low","close"]].astype(float)
    api_df["volume"] = api_df["volume"].astype("int64")
except (requests.RequestException, ValueError, KeyError) as exc:
    raise RuntimeError(f"API ingestion failed: {exc}") from exc
api_df.head()


        date    open      high      low   close   volume
0 2026-08-20  236.00  237.5000  233.030  233.69  4028353
1 2026-08-19  232.26  238.6150  230.510  237.16  5342567
2 2026-08-18  229.73  234.4499  229.415  232.67  4650116
3 2026-08-17  231.59  233.6500  227.400  228.85  7564317
4 2026-08-14  238.37  239.1500  233.730  234.32  4523467

In [3]:
required_api = {"date","open","high","low","close","volume"}
assert required_api.issubset(api_df.columns)
assert api_df[list(required_api)].isna().sum().sum() == 0
assert (api_df["high"] >= api_df["low"]).all()
assert (api_df["volume"] >= 0).all()
api_path = RAW_DIR / f"api_alphavantage_IBM_{RUN_TS}.csv"
api_df.to_csv(api_path, index=False)
print("API shape:", api_df.shape)
print("API missing values:", int(api_df.isna().sum().sum()))
print("Saved:", api_path)


API shape: (100, 6)
API missing values: 0
Saved: homework/homework04/data/raw/api_alphavantage_IBM_20260820-0000.csv


## 2. Scrape a small table


In [4]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (educational assignment)"}
try:
    response = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.select_one("table#constituents")
    if table is None:
        raise ValueError("S&P 500 constituents table was not found")
    rows = []
    for tr in table.select("tbody tr")[:10]:
        cells = [cell.get_text(" ", strip=True) for cell in tr.select("td")]
        if len(cells) >= 8:
            rows.append(cells[:8])
    scrape_df = pd.DataFrame(rows, columns=["symbol","security","gics_sector",
        "gics_sub_industry","headquarters","date_added","cik","founded"])
    scrape_df["date_added"] = pd.to_datetime(scrape_df["date_added"], errors="coerce")
    scrape_df["cik"] = scrape_df["cik"].astype("string").str.zfill(10)
except (requests.RequestException, ValueError) as exc:
    raise RuntimeError(f"Table scraping failed: {exc}") from exc
scrape_df.head()


  symbol              security             gics_sector
0    MMM                    3M              Industrials
1    AOS          A. O. Smith              Industrials
2    ABT   Abbott Laboratories              Health Care
3   ABBV                AbbVie              Health Care
4    ACN             Accenture  Information Technology

In [5]:
required_scrape = {"symbol","security","gics_sector","headquarters","cik"}
assert required_scrape.issubset(scrape_df.columns)
assert scrape_df[list(required_scrape)].isna().sum().sum() == 0
assert scrape_df["symbol"].str.len().gt(0).all()
assert scrape_df["cik"].str.fullmatch(r"\d{10}").all()
scrape_path = RAW_DIR / f"scrape_wikipedia_sp500_{RUN_TS}.csv"
scrape_df.to_csv(scrape_path, index=False)
print("Scrape shape:", scrape_df.shape)
print("Missing values:", int(scrape_df.isna().sum().sum()))
print("Saved:", scrape_path)


Scrape shape: (10, 8)
Missing values: 0
Saved: homework/homework04/data/raw/scrape_wikipedia_sp500_20260820-0000.csv


## Assumptions and risks

- The Alpha Vantage demo key is rate-limited; unexpected responses become clear errors.
- API values may change on rerun, so timestamped files preserve acquisition history.
- Wikipedia is public, but its HTML can change; the scraper uses the table ID and validates fields.
- Only the first 10 component rows are retained to keep the scrape small.
- Invalid dates become missing values and are reported by validation.
